# 03 — Why naive RAG fails on these questions

This experiment runs the same frozen questions as notebooks 02 and 04 through vector retrieval over a **fixed random sample of 50 invoice lines**. It is intentionally not a table-wide aggregation system: nearest-neighbor retrieval only supplies a few matching rows.

The first run embeds the sample with Gemini and caches vectors under ignored `data/processed/`. Later runs reuse the cache. Six question embeddings and one Gemini generation call are made each run. Expected answers are used only for scoring, never sent to the RAG model.

In [2]:
import hashlib
import json
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from src.evaluation import load_gold_set, score_answer
from src.gemini import build_gemini_llm, get_gemini_api_key, load_project_env

load_project_env(ROOT / ".env")
# print(f"Repo root: {ROOT}")
print(f"Gemini API key present: {bool(os.getenv('GOOGLE_API_KEY'))}")

Gemini API key present: True


## Prepare the fixed invoice-line sample

A deterministic sample keeps embedding use under the Gemini free-tier embedding quota and makes the experiment repeatable. Because only 50 of roughly 525,000 rows are indexed, this corpus cannot reliably answer whole-dataset totals, means, or maxima.

In [3]:
DATA_PATH = ROOT / "data" / "processed" / "online_retail_2009_2010.pkl"
GOLD_PATH = ROOT / "eval" / "gold_generation.jsonl"
CACHE_PATH = ROOT / "data" / "processed" / "naive_rag_sample_embeddings.npz"
SAMPLE_SIZE = 50
RANDOM_SEED = 42
TOP_K = 5
EMBEDDING_MODEL = os.getenv("GEMINI_EMBED_MODEL", "gemini-embedding-2-preview")

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Processed dataset not found: {DATA_PATH}. Run notebook 01 first.")

df = pd.read_pickle(DATA_PATH)
gold_set = load_gold_set(GOLD_PATH)
sample = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED).sort_index()

def format_invoice_line(row):
    customer_id = "missing" if pd.isna(row["Customer ID"]) else str(int(row["Customer ID"]))
    description = "missing" if pd.isna(row["Description"]) else str(row["Description"])
    return (
        f"invoice={row['Invoice']}; date={row['InvoiceDate']}; country={row['Country']}; "
        f"description={description}; stock_code={row['StockCode']}; "
        f"quantity={int(row['Quantity'])}; unit_price={float(row['Price']):.2f}; "
        f"revenue={float(row['Revenue']):.2f}; customer_id={customer_id}; "
        f"cancellation={bool(row['is_cancellation'])}; guest={bool(row['is_guest'])}"
    )

documents = [format_invoice_line(row) for _, row in sample.iterrows()]
source_indices = sample.index.to_numpy(dtype=np.int64)
print(f"Indexed sample: {len(documents):,} of {len(df):,} invoice lines ({len(documents) / len(df):.2%})")
display(pd.DataFrame({"source_row": source_indices[:5], "invoice_line": documents[:5]}))

Indexed sample: 50 of 525,461 invoice lines (0.01%)


,source_row,invoice_line
0,4182,invoice=489801; date=2009-12-02 12:28:00; coun...
1,9377,invoice=490136; date=2009-12-03 19:13:00; coun...
2,22373,invoice=491155; date=2009-12-10 09:49:00; coun...
3,22377,invoice=491155; date=2009-12-10 09:49:00; coun...
4,23742,invoice=491240; date=2009-12-10 14:50:00; coun...


## Embed the sample or load its local cache

The cache key includes the embedding model and exact sampled invoice text. It is regenerated when either changes. The embedding batch size is capped at 100 rows per request.

In [5]:
cache_key = hashlib.sha256(
    (EMBEDDING_MODEL + "\n" + "\n".join(documents)).encode("utf-8")
).hexdigest()

embed_model = GoogleGenAIEmbedding(
    model_name=EMBEDDING_MODEL,
    api_key=get_gemini_api_key(),
    embed_batch_size=100,
)

cache_is_valid = False
if CACHE_PATH.is_file():
    with np.load(CACHE_PATH, allow_pickle=False) as cache:
        cache_is_valid = str(cache["cache_key"].item()) == cache_key
        if cache_is_valid:
            embeddings = np.asarray(cache["embeddings"], dtype=np.float32)
            cached_documents = cache["documents"].astype(str).tolist()
            if cached_documents != documents:
                raise ValueError("Embedding cache key matches but cached rows differ. Remove the cache and retry.")

if cache_is_valid:
    print(f"Loaded {len(embeddings):,} cached embeddings")
else:
    print("Embedding the fixed invoice-line sample; this only happens when the cache is missing or outdated.")
    embeddings = np.asarray(
        embed_model.get_text_embedding_batch(documents, show_progress=True),
        dtype=np.float32,
    )
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    with CACHE_PATH.open("wb") as cache_file:
        np.savez_compressed(
            cache_file,
            embeddings=embeddings,
            documents=np.asarray(documents, dtype=str),
            source_indices=source_indices,
            cache_key=np.asarray(cache_key),
        )
    print(f"Saved embeddings to {CACHE_PATH}")

if embeddings.ndim != 2 or embeddings.shape[0] != len(documents):
    raise ValueError(f"Unexpected embedding matrix shape: {embeddings.shape}")
if not np.isfinite(embeddings).all():
    raise ValueError("Embedding matrix contains non-finite values.")

corpus_norms = np.linalg.norm(embeddings, axis=1)
if np.any(corpus_norms == 0):
    raise ValueError("Embedding matrix contains a zero-length vector.")
normalized_embeddings = embeddings / corpus_norms[:, None]

Loaded 50 cached embeddings


## Retrieve the nearest invoice lines

Embed each benchmark question as a retrieval query, then rank the sampled rows by cosine similarity. These top-k rows are the entire evidence provided to the generation step for that question.

In [6]:
retrieval_by_id = {}
for item in gold_set:
    query_vector = np.asarray(embed_model.get_query_embedding(item["question"]), dtype=np.float32)
    query_norm = np.linalg.norm(query_vector)
    if query_norm == 0:
        raise ValueError(f"Question embedding has zero length: {item['id']}")
    similarities = normalized_embeddings @ (query_vector / query_norm)
    top_indices = np.argsort(similarities)[-TOP_K:][::-1]
    retrieval_by_id[item["id"]] = [
        {
            "source_row": int(source_indices[index]),
            "similarity": float(similarities[index]),
            "text": documents[index],
        }
        for index in top_indices
    ]

retrieval_preview = pd.DataFrame([
    {
        "question_id": item["id"],
        "rank": rank,
        "similarity": round(hit["similarity"], 4),
        "source_row": hit["source_row"],
        "invoice_line": hit["text"],
    }
    for item in gold_set
    for rank, hit in enumerate(retrieval_by_id[item["id"]], start=1)
])
display(retrieval_preview)

INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "

,question_id,rank,similarity,source_row,invoice_line
0,top_gross_customer,1,0.6460,179482,invoice=506410; date=2010-04-29 13:54:00; coun...
1,top_gross_customer,2,0.6386,220927,invoice=510819; date=2010-06-03 19:47:00; coun...
2,top_gross_customer,3,0.6380,71436,invoice=495809; date=2010-01-27 09:44:00; coun...
3,top_gross_customer,4,0.6351,391453,invoice=527050; date=2010-10-14 12:54:00; coun...
4,top_gross_customer,5,0.6332,359910,invoice=524270; date=2010-09-28 11:50:00; coun...
5,average_net_clv,1,0.6204,359910,invoice=524270; date=2010-09-28 11:50:00; coun...
6,average_net_clv,2,0.6179,220927,invoice=510819; date=2010-06-03 19:47:00; coun...
7,average_net_clv,3,0.6099,71436,invoice=495809; date=2010-01-27 09:44:00; coun...
8,average_net_clv,4,0.6088,344329,invoice=522968; date=2010-09-19 13:05:00; coun...
9,average_net_clv,5,0.6077,251334,invoice=513627; date=2010-06-27 13:16:00; coun...


## Generate answers using retrieved context only

All six questions and their retrieved snippets are sent in one Gemini call to reduce request overhead. The prompt explicitly says the retrieved lines are an incomplete sample and asks the model not to claim an exact whole-dataset statistic without evidence. Ground truth is not included in the prompt.

In [8]:
rag_cases = [
    {
        "case_number": case_number,
        "question": item["question"],
        "retrieved_invoice_lines": [
            {"similarity": round(hit["similarity"], 4), "text": hit["text"]}
            for hit in retrieval_by_id[item["id"]]
        ],
    }
    for case_number, item in enumerate(gold_set, start=1)
]
rag_prompt = (
    "Answer each question using only its retrieved invoice lines. The lines come from a small random sample of a much larger transaction table and are incomplete evidence. Do not claim that a global total, mean, maximum, or ranking is exact based on these few rows. If the retrieved evidence cannot establish the requested result, say that it is insufficient. Return only valid JSON with this shape: "
    '{"results":[{"case_number":1,"answer":"..."}]}. Return one result per case in the supplied order, using the same case_number. No markdown fences or extra text.\n\n'
    + json.dumps(rag_cases, ensure_ascii=False)
)

rag_llm = build_gemini_llm(temperature=0.0, max_tokens=1024)
rag_response = str(await rag_llm.acomplete(rag_prompt)).strip()
if rag_response.startswith("```"):
    rag_response = re.sub(r"^```(?:json)?\s*|\s*```$", "", rag_response, flags=re.IGNORECASE)
rag_payload = json.loads(rag_response)
if not isinstance(rag_payload, dict) or not isinstance(rag_payload.get("results"), list):
    raise ValueError("RAG model response must be a JSON object with a results list.")

if len(rag_payload["results"]) != len(gold_set):
    raise ValueError(f"RAG model returned {len(rag_payload['results'])} results for {len(gold_set)} questions.")
answers_by_case = {}
for result in rag_payload["results"]:
    if not isinstance(result, dict) or not isinstance(result.get("case_number"), int):
        raise ValueError("Each RAG answer must have an integer case_number.")
    case_number = result["case_number"]
    if case_number < 1 or case_number > len(gold_set):
        raise ValueError(f"RAG model returned an unknown case_number: {case_number}")
    if case_number in answers_by_case:
        raise ValueError(f"RAG model returned duplicate case_number: {case_number}")
    if not isinstance(result.get("answer"), str):
        raise ValueError(f"RAG answer must be text for case {case_number}.")
    answers_by_case[case_number] = result["answer"]
expected_cases = set(range(1, len(gold_set) + 1))
if set(answers_by_case) != expected_cases:
    raise ValueError(f"RAG case numbers do not match benchmark: {sorted(answers_by_case)}")
answers_by_id = {
    item["id"]: answers_by_case[case_number]
    for case_number, item in enumerate(gold_set, start=1)
}

rag_results = []
for item in gold_set:
    answer = answers_by_id[item["id"]]
    rag_results.append({
        "id": item["id"],
        "question": item["question"],
        "rag_answer": answer,
        "expected_answer": item["expected_display"],
        "rag_match": score_answer(answer, item)["passed"],
        "top_retrieved_rows": len(retrieval_by_id[item["id"]]),
    })

rag_comparison = pd.DataFrame(rag_results)
display(rag_comparison)
rag_accuracy = rag_comparison["rag_match"].mean()
print(f"Naive RAG exact/tolerance match: {rag_accuracy:.1%} ({rag_comparison['rag_match'].sum()}/{len(rag_comparison)})")

INFO:httpx:HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite "HTTP/1.1 200 OK"
HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite "HTTP/1.1 200 OK"
INFO:google_genai.chats:AFC is enabled with max remote calls: 10.
AFC is enabled with max remote calls: 10.


,id,question,rag_answer,expected_answer,rag_match,top_retrieved_rows
0,top_gross_customer,Which identified customer generated the most g...,"Based on the retrieved evidence, customer 1821...",18102,False,5
1,average_net_clv,What was the average net customer lifetime val...,The retrieved evidence is insufficient to esta...,"1,950.56",False,5
2,top_country_revenue,Which country generated the most net revenue a...,"Based on the retrieved lines, United Kingdom i...",United Kingdom,True,5
3,largest_line_revenue,What was the largest single positive line reve...,"Based on the retrieved lines, the largest sing...","25,111.09",False,5
4,identified_customer_count,How many identified customers made at least on...,"Based on the retrieved invoice lines, there ar...","4,312",False,5
5,total_net_revenue,What was total net revenue across all transact...,"Based on the retrieved evidence, the sum of re...","9,539,484.63",False,5


Naive RAG exact/tolerance match: 16.7% (1/6)


## What this comparison shows

The gold answer and scoring rules are identical to the table-Q&A evaluation. A low RAG score is expected: retrieving five semantically related invoice lines does not provide the full population required for a customer maximum, mean lifetime value, country ranking, or total. RAG remains useful for finding relevant examples or answering questions about specific rows; exact global analytics need a query engine that can aggregate the table.